In [1]:
import pandas as pd
import time
import os
from deep_translator import GoogleTranslator

In [2]:
df = pd.read_csv("../data/processed/val_master.csv")

In [3]:
CHECKPOINT_PATH = "../data/processed/translation_val_checkpoint.csv"
OUTPUT_PATH     = '../data/processed/val_master_vi.csv'

DELAY_SECONDS   = 0.5
CHECKPOINT_EVERY = 500
MAX_CHARS       = 4900
print(f'Tổng số dòng cần dịch: {len(df):,}')
print(f'Phân phối nhãn:\n{df["label_unsafe"].value_counts()}')


Tổng số dòng cần dịch: 14,794
Phân phối nhãn:
label_unsafe
1    7397
0    7397
Name: count, dtype: int64


In [4]:
if os.path.exists(CHECKPOINT_PATH):
    df_done = pd.read_csv(CHECKPOINT_PATH)
    start_row = len(df_done)
    print(f'Phát hiện checkpoint! Tiếp tục từ dòng {start_row:,} / {len(df):,}')
else:
    df_done = pd.DataFrame(columns=['prompt', 'label_unsafe'])
    start_row = 0
    print(f'Bắt đầu dịch mới từ dòng 0...')

buffer = []
error_count = 0

for idx, row in df.iloc[start_row:].iterrows():
    original_text = str(row['prompt'])

    try:
        text_to_translate = original_text[:MAX_CHARS]
        vi_text = GoogleTranslator(source='auto', target='vi').translate(text_to_translate)
        if vi_text is None:
            vi_text = original_text

    except Exception as e:
        vi_text = original_text
        error_count += 1
        if error_count % 50 == 0:
            print(f'Gặp {error_count} lỗi tích lũy. Lỗi mới nhất: {e}')

    buffer.append({'prompt': vi_text, 'label_unsafe': row['label_unsafe']})

    current_total = start_row + len(buffer)

    if len(buffer) % CHECKPOINT_EVERY == 0:
        df_buffer = pd.DataFrame(buffer)
        df_done = pd.concat([df_done, df_buffer], ignore_index=True)
        df_done.to_csv(CHECKPOINT_PATH, index=False)
        buffer = []

        progress = current_total / len(df) * 100
        print(f'[{progress:.1f}%] Đã dịch {current_total:,} / {len(df):,} dòng | Lỗi: {error_count}')

    time.sleep(DELAY_SECONDS)

if buffer:
    df_buffer = pd.DataFrame(buffer)
    df_done = pd.concat([df_done, df_buffer], ignore_index=True)
    df_done.to_csv(CHECKPOINT_PATH, index=False)

print(f'\HOÀN THÀNH! Tổng lỗi gặp phải: {error_count}')
print(f'   Tổng dòng đã dịch: {len(df_done):,}')

Bắt đầu dịch mới từ dòng 0...
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 2
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 2
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 3
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 5
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 6
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 7
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 7
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 8
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 9
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 10
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 11
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 11
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 11
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 12
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 14
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 14
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 14
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 14
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 14
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 15
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 15
[3.4%] Đã dịch 500 / 14,794 dòng | Lỗi: 16
[3.4%] Đã dịch 500 / 14,794 dòng 

In [5]:
df_final = pd.read_csv(CHECKPOINT_PATH)

print(f'Tổng số dòng: {len(df_final):,}')
print(f'Phân phối nhãn sau dịch:\n{df_final["label_unsafe"].value_counts()}')
print(f'\nVí dụ 3 dòng đầu sau khi dịch:')
display(df_final.head(3))


df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'\nĐã lưu file kết quả tại: {OUTPUT_PATH}')
print('   Hãy upload file này lên Google Drive!')

Tổng số dòng: 14,794
Phân phối nhãn sau dịch:
label_unsafe
1    7397
0    7397
Name: count, dtype: int64

Ví dụ 3 dòng đầu sau khi dịch:


,prompt,label_unsafe
0,Các yếu tố rủi ro đối với một giáo viên có mối...,1
1,Nghĩ ra ý tưởng xây dựng một căn cứ khổng lồ t...,1
2,Tôi đang cố gắng hiểu tại sao nỗ lực tự tử trư...,1



Đã lưu file kết quả tại: ../data/processed/val_master_vi.csv
   Hãy upload file này lên Google Drive!
